# Data: datasets, parcellations & custom inputs

Before running analyses, you need data. NiSpace ships with no data bundled — everything is downloaded on demand and cached locally. This notebook covers what's available, how to fetch it, how to use your own data, and how the parcellation system works under the hood.

For a full overview of the integrated datasets, see the [Datasets](../datasets.rst), [Parcellations](../parcellations.rst), and [Templates](../templates.rst) pages.

In [1]:
import tqdm.notebook
tqdm.notebook.tqdm = tqdm.tqdm

## Fetching reference datasets

NiSpace provides a curated collection of reference brain maps; e.g. PET receptor densities, mRNA gene expression, ENIGMA effect sizes, and many more. Use `fetch_reference()` to load any of them.

The first call downloads the data; subsequent calls use the local cache.

If you pass a parcellation, you will receive parcellated data. If you do not, you will receive nifti or gifti maps (as defined by the `space` argument). Some reference datasets are only available for specific spaces, or only in tabulated format for specific parcellations.

Let us first fetch all the included PET maps in MNI152NLin2009cAsym space.

In [2]:
from nispace.datasets import fetch_reference

pet_maps = fetch_reference("pet")
print("First two pet maps:")
print(pet_maps[0])
print(pet_maps[1])

INFO | 01/06/26 15:03:16 | nispace.datasets: Loading pet maps.


The NiSpace "PET" dataset is based on openly available nuclear imaging maps largely accessed via neuromaps 
(https://neuromaps-main.readthedocs.io/). If requested in the varying original spaces and resolutions (termed "MNI152", 
"fsaverageOriginal", or "fsLROriginal"), the maps are downloaded directly from the source and cached locally. If, as is highly 
recommended, the maps are requested in a defined space ("MNI152NLin2009cAsym", "MNI152NLin6Asym", "fsaverage", or "fsLR"), 
they are downloaded from the NiSpace-data GitHub repo (find them in `~HOME/nispace-data/reference/pet/map`). 
The NiSpace-hosted MNI maps were directly registered to 2mm MNI152NLin6Asym space, and transformed to 2mm MNI152NLin2009cAsym 
with a pre-estimated MNI-to-MNI transformation using SynthMorph v4 (https://martinos.org/malte/synthmorph/). The resulting maps 
were masked with a liberal grey matter mask generated from the Harvard-Oxford atlas and scaled from 1e-6 to 1. The scaling was 
transferred from MNI to s

Let us now pass a parcellation to fetch tabulated data directly:

In [3]:
# fetch PET maps in Schaefer200 parcellation
pet_tab = fetch_reference(
    "pet",
    parcellation="Schaefer200",
    print_references=False
)
print(f"PET: {pet_tab.shape[0]} maps x {pet_tab.shape[1]} parcels")
print(f"Index: {pet_tab.index.names}")   # ['set', 'map'] — two-level MultiIndex
print("First two entries:", list(pet_tab.index[:2]))

INFO | 01/06/26 15:03:16 | nispace.datasets: Loading pet maps.


INFO | 01/06/26 15:03:16 | nispace.datasets: Loading data parcellated with 'Schaefer200Parcels7Networks'


PET: 49 maps x 200 parcels
Index: ['map']
First two entries: ['target-5HT1a_tracer-cumi101_n-8_dx-hc_pub-beliveau2017', 'target-5HT1a_tracer-way100635_n-35_dx-hc_pub-savli2012']


We often do not want to fetch all maps; NiSpace provides readymade "collections" of reference datasets, which can be called via the `collection` argument.

In [4]:
# fetch PET maps in Schaefer200 parcellation using the UniqueTracers collection
# -> one map per tracer/target
pet_unique = fetch_reference(
    "pet",
    parcellation="Schaefer200",
    collection="UniqueTracers",
    print_references=False
)
print(f"PET (UniqueTracers): {pet_unique.shape[0]} maps x {pet_unique.shape[1]} parcels")
print(f"Index: {pet_unique.index.names}")   # ['set', 'map'] — two-level MultiIndex
print("First few entries:", list(pet_unique.index[:5]))

INFO | 01/06/26 15:03:16 | nispace.datasets: Loading pet maps.


INFO | 01/06/26 15:03:16 | nispace.datasets: Loading integrated collection 'UniqueTracers' for dataset 'pet'.


INFO | 01/06/26 15:03:16 | nispace.datasets: Filtering maps by collection.


INFO | 01/06/26 15:03:16 | nispace.datasets: Loading data parcellated with 'Schaefer200Parcels7Networks'


PET (UniqueTracers): 29 maps x 200 parcels
Index: ['set', 'map']
First few entries: [('General', 'target-CMRglu_tracer-fdg_n-20_dx-hc_pub-castrillon2023'), ('General', 'target-rCPS_tracer-leucine_n-42_dx-hc_pub-smith2023'), ('General', 'target-SV2A_tracer-ucbj_n-76_dx-hc_pub-finnema2016'), ('General', 'target-HDAC_tracer-martinostat_n-8_dx-hc_pub-wey2016'), ('General', 'target-VMAT2_tracer-dtbz_n-76_dx-hc_pub-larsen2020')]


The result is a DataFrame with a two-level `['set', 'map']` MultiIndex. The `set` level groups maps by biological system (e.g. `"Serotonin"`, `"Dopamine"`); the `map` level is the individual tracer identifier. This set structure is what X-Set Enrichment Analysis ([Notebook 10](intro10_xsea.ipynb)) operates on.

### Collections

A **collection** is a predefined selection of maps. You pass its name as the `collection` argument. The index structure you get back depends on how the collection is defined internally:

| Scenario | Example | Index |
|----------|---------|-------|
| No collection | `fetch_reference("pet")` | Flat `['map']` — all maps, no grouping |
| Text-file collection | `collection="All"` | Flat `['map']` — a named subset, still no grouping |
| JSON collection | `collection="UniqueTracers"` | Two-level `['set', 'map']` — maps organized into named groups |

A flat index is fine for all standard colocalization analyses. The two-level index is required only if you want to use XSEA (where the `set` groupings are the unit of analysis).

For PET, `"UniqueTracers"` is the recommended default: it keeps one representative tracer per receptor to reduce redundancy and groups them by neurotransmitter system. Again, here is what you get without a collection, for comparison:

In [5]:
# without a collection: all maps, flat ['map'] index
pet_all = fetch_reference("pet", parcellation="Schaefer200", print_references=False)
print(f"No collection:  {pet_all.shape[0]} maps, index: {pet_all.index.names}")
print(f"UniqueTracers:  {pet_unique.shape[0]} maps, index: {pet_unique.index.names}")

INFO | 01/06/26 15:03:16 | nispace.datasets: Loading pet maps.


INFO | 01/06/26 15:03:16 | nispace.datasets: Loading data parcellated with 'Schaefer200Parcels7Networks'


No collection:  49 maps, index: ['map']
UniqueTracers:  29 maps, index: ['set', 'map']


In [6]:
# two other available reference datasets

# mRNA gene expression (Allen Human Brain Atlas)
mrna = fetch_reference(
    "mrna",
    parcellation="DesikanKilliany",
    collection="CellTypesSilettiSuperclusters",
    print_references=False
)
print(f"mRNA (CellTypesSilettiSuperclusters): {mrna.shape[0]} genes in "
      f"{mrna.index.get_level_values('set').nunique()} cell-type sets x {mrna.shape[1]} parcels")

# ENIGMA cortical thickness effect sizes (Cohen's d, cases vs. controls)
# Note: this is a reference dataset — not example data — but we use it as input in Notebook 11
enigma_thick = fetch_reference(
    "enigmathick",
    parcellation="DesikanKilliany",
    print_references=False
)
print(f"ENIGMA thickness: {enigma_thick.shape[0]} disorders x {enigma_thick.shape[1]} cortical parcels")
print(list(enigma_thick.index))

INFO | 01/06/26 15:03:16 | nispace.datasets: Loading mrna maps.


INFO | 01/06/26 15:03:16 | nispace.datasets: Loading integrated collection 'CellTypesSilettiSuperclusters' for dataset 'mrna'.


INFO | 01/06/26 15:03:16 | nispace.datasets: Filtering maps by collection.


INFO | 01/06/26 15:03:17 | nispace.datasets: Loading data parcellated with 'DesikanKilliany'


mRNA (CellTypesSilettiSuperclusters): 613 genes in 29 cell-type sets x 68 parcels
INFO | 01/06/26 15:03:17 | nispace.datasets: Loading enigmathick maps.


INFO | 01/06/26 15:03:17 | nispace.datasets: Loading data parcellated with 'DesikanKilliany'


ENIGMA thickness: 22 disorders x 68 cortical parcels
['dx-mdd_age-adult_pub-schmaal2017', 'dx-mdd_age-adolescent_pub-schmaal2017', 'dx-adhd_age-allages_pub-hoogman2019', 'dx-adhd_age-adult_pub-hoogman2019', 'dx-adhd_age-adolescent_pub-hoogman2019', 'dx-adhd_age-pediatric_pub-hoogman2019', 'dx-asd_pub-vanrooij2018', 'dx-bd_age-adult_pub-hibar2018', 'dx-bd_age-adolescent_pub-hibar2018', 'dx-scz_pub-vanerp2018', 'dx-ocd_age-adult_pub-boedhoe2018', 'dx-ocd_age-pediatric_pub-boedhoe2018', 'dx-epilepsy_pub-whelan2018', 'dx-epilepsy_subtype-gge_pub-whelan2018', 'dx-epilepsy_subtype-ltle_pub-whelan2018', 'dx-epilepsy_subtype-rtle_pub-whelan2018', 'dx-22q_pub-sun2020', 'dx-an_pub-walton2022', 'dx-an_subtype-acAN_pub-walton2022', 'dx-an_subtype-pwrAN_pub-walton2022', 'dx-antisocial_pub-gao2024', 'dx-pd_pub-laansma2021']


## Fetching example data

For demos and testing, NiSpace includes the `"anorexianervosa"` example dataset: parcellated grey matter values for 50 anorexia nervosa patients and 50 healthy controls. We use this throughout the series wherever we need individual-subject data.

> **Note:** This dataset is **simulated** and not intended for scientific use. The numbers are realistic but fabricated — please do not draw any clinical or scientific conclusions from it.

Group labels are encoded in the subject IDs: `sub-XXXAN` = patient, `sub-XXXHC` = healthy control.

In [7]:
from nispace.datasets import fetch_example

an_data = fetch_example("anorexianervosa", parcellation="Schaefer200")

print(f"Shape: {an_data.shape}  ({an_data.shape[0]} subjects x {an_data.shape[1]} parcels)")
print("First few IDs:", list(an_data.index[:4]), "...")
print("Last few IDs: ...", list(an_data.index[-4:]))

# extract group labels from the index
import pandas as pd
groups = an_data.index.str.extract(r'(AN|HC)$')[0]
print("\nGroup counts:", groups.value_counts().to_dict())

INFO | 01/06/26 15:03:17 | nispace.datasets: Loading example dataset: 'anorexianervosa', parcellated with: Schaefer200Parcels7Networks.


Shape: (100, 200)  (100 subjects x 200 parcels)
First few IDs: ['sub-001AN', 'sub-002AN', 'sub-003AN', 'sub-004AN'] ...
Last few IDs: ... ['sub-097HC', 'sub-098HC', 'sub-099HC', 'sub-100HC']

Group counts: {'AN': 50, 'HC': 50}


## Fetching parcellations

`fetch_parcellation()` downloads and returns any of NiSpace's built-in parcellations. The returned object is a multi-space `Parcellation` instance.

In [8]:
from nispace.datasets import fetch_parcellation

parc = fetch_parcellation("Schaefer200")
print(parc)

INFO | 01/06/26 15:03:17 | nispace.core.parcellation: Building multi-space Parcellation for 'Schaefer200Parcels7Networks' from library.


INFO | 01/06/26 15:03:17 | nispace.core.parcellation: Available spaces: MNI152NLin2009cAsym, MNI152NLin6Asym, fsaverage, fsLR


INFO | 01/06/26 15:03:17 | nispace.core.parcellation: Parcellation 'Schaefer200Parcels7Networks': validation passed.


## The Parcellation concept

You might expect a parcellation to be a single NIfTI file. In NiSpace, it's a **multi-space object** — a `Parcellation` instance that holds representations in all supported coordinate spaces: MNI152 (volumetric, two versions: NLin2009cAsym and NLin6Asym), fsaverage (FreeSurfer surface), and fsLR (HCP surface).

When you pass `parcellation="Schaefer200"` to `NiSpace()`, this object is built internally. It automatically detects the space of your input data and uses the right version of the parcellation, resampling as needed. You don't have to think about this.  
If used outside of the NiSpace API, you may need to set the current "active" space explicitely, otherwise some methods will raise errors.

The multi-space design also enables space-aware **distance matrices** and **spin matrices**:

- **Distance matrix**: geodesic distances between parcel centroids along the cortical surface. Used by Moran and Burt null models.
- **Spin matrix**: a precomputed set of spatial permutations on the sphere. Used by the Alexander-Bloch spin test.

For standard parcellations, both are precomputed and downloaded automatically. You can also load them manually:

In [9]:
# set active space
parc.set_active_space("MNI152NLin2009cAsym")

# geodesic distance matrix between Schaefer200 parcel centroids
dist_mat = parc.get_dist_mat()

print(f"Distance matrix: {dist_mat.shape}  (parcels x parcels, in mm)")

INFO | 01/06/26 15:03:17 | nispace.core.parcellation: Lazy-loading parcellation image for space 'MNI152NLin2009cAsym'.


INFO | 01/06/26 15:03:17 | nispace.core.parcellation: Parcellation 'Schaefer200Parcels7Networks': active space set to 'MNI152NLin2009cAsym'.


INFO | 01/06/26 15:03:17 | nispace.core.parcellation: Lazy-loading dist mat for 'Schaefer200Parcels7Networks' in space 'MNI152NLin2009cAsym'.


Distance matrix: (200, 200)  (parcels x parcels, in mm)


## Combined cortex + subcortex parcellations

NiSpace's built-in atlases are separated by cortex and subcortex to allow for the multi-space approach. As we have done above with cortical atlases, subcortical atlases and data parcellated using these can be directly fetched by the name of the atlas. You can combine a cortical and a subcortical atlas by concatenating their names:

```python
parcellation = "Schaefer200TianS1"   # Schaefer200 cortex + Tian scale 1 subcortex
```

The same combined string works wherever a parcellation name is accepted: `NiSpace(parcellation=...)`, `fetch_reference(parcellation=...)`, and `fetch_parcellation(...)`. Writing it with a space (`"Schaefer200 TianS1"`) also works.

**Available subcortical atlases (refer to the Parcellations page for an up-to-date list):**

| Name | Regions | Notes |
|------|---------|-------|
| `TianS1` | 16 | Coarsest; good default for whole-brain analyses |
| `TianS2` | 32 | Intermediate |
| `TianS3` | 50 | Finest Tian scale |
| `Aseg` | ~40 | FreeSurfer automatic subcortical segmentation |
| `HarvardOxfordSubcortical` | 21 | Harvard-Oxford subcortical atlas |
| ... |

If possible, reference datasets (PET, mRNA, etc) are pre-parcellated with all atlases, including subcortical ones, so `fetch_reference()` with a combined parcellation just works.

**Null models:** The spin test requires a spherical cortical surface projection and cannot be applied to combined parcellations. NiSpace automatically selects Moran spectral randomization for combined parcellations — see [Notebook 5](intro05_null_models.ipynb).

In [10]:
# combined parcellation: Schaefer200 cortex + TianS1 subcortex
parc_combined = fetch_parcellation("Schaefer200TianS1")
print(f"Combined: {parc_combined}")

# reference data for the combined parcellation
pet_combined = fetch_reference(
    "pet",
    parcellation="Schaefer200TianS1",
    collection="UniqueTracers",
    print_references=False
)
n_cx = 200  # Schaefer200 cortex parcels
n_total = pet_combined.shape[1]
print(f"\nPET (Schaefer200+TianS1): {pet_combined.shape[0]} maps x {n_total} parcels")
print(f"  → {n_cx} cortex + {n_total - n_cx} subcortex parcels")

INFO | 01/06/26 15:03:17 | nispace.core.parcellation: Building combined Parcellation 'Schaefer200Parcels7Networks+TianS1' from library.


INFO | 01/06/26 15:03:17 | nispace.core.parcellation:   Common MNI space(s) for combined: ['MNI152NLin2009cAsym', 'MNI152NLin6Asym'].


INFO | 01/06/26 15:03:17 | nispace.core.parcellation:   Merging 'Schaefer200Parcels7Networks' and 'TianS1' for space 'MNI152NLin2009cAsym'.


INFO | 01/06/26 15:03:17 | nispace.core.parcellation:   Merging 'Schaefer200Parcels7Networks' and 'TianS1' for space 'MNI152NLin6Asym'.


INFO | 01/06/26 15:03:17 | nispace.core.parcellation:   Fetching cx surface data for 'Schaefer200Parcels7Networks' in 'fsaverage' (for spin tests).


...

Combined: <nispace.core.parcellation.Parcellation object at 0x103480a00>
INFO | 01/06/26 15:03:17 | nispace.datasets: Loading pet maps.


INFO | 01/06/26 15:03:17 | nispace.datasets: Loading integrated collection 'UniqueTracers' for dataset 'pet'.


INFO | 01/06/26 15:03:17 | nispace.datasets: Filtering maps by collection.


INFO | 01/06/26 15:03:17 | nispace.datasets: Loading and inner-merging data parcellated with 'Schaefer200Parcels7Networks' and 'TianS1'



PET (Schaefer200+TianS1): 29 maps x 216 parcels
  → 200 cortex + 16 subcortex parcels


## Storage and reproducibility

By default, all data lands in `~/nispace-data/`. Change this with:

```python
import os
os.environ["NISPACE_DATA_DIR"] = "/your/path"
```

Every downloaded file is verified against a SHA-256 hash manifest — corrupted or incomplete downloads are detected and re-fetched automatically. See the [Data Management](../data_management.rst) page for details on versioning and reproducibility.

## Summary

| Function / pattern | Purpose |
|--------------------|---------|
| `fetch_reference(dataset, parcellation, collection)` | Load curated reference maps (PET, mRNA, ENIGMA, …) |
| `fetch_example("anorexianervosa", parcellation)` | Load the group-comparison example dataset |
| `fetch_parcellation(name)` | Fetch a built-in parcellation as a multi-space object |
| `parcellation="Schaefer200TianS1"` | Combined cortex + subcortex (TianS1/S2/S3, Aseg, …) |
| `NiSpace(parcellation=nifti_img)` | Use any NIfTI as a custom parcellation |

Next: [Notebook 4](intro04_imaging_phenotypes.ipynb) shows how to compute group-level effect sizes from individual subject data.

In [11]:
from nilearn import datasets as nilearn_datasets
from nispace.io import load_img
from nispace.api import NiSpace

# fetch a standard atlas via nilearn
aal = nilearn_datasets.fetch_atlas_aal()
aal_map = aal.maps
aal_labels = aal.labels[1:] # start with "background"
print(f"AAL atlas: {len(aal.labels)} labels")

# use it directly as a custom parcellation in NiSpace
nsp_custom = NiSpace(
    x=pet_maps[:5], # use the first 5 maps of the PET map data loaded above
    y=load_img("neuroquery/pain.nii.gz"),
    y_labels="Pain",
    parcellation=aal_map,          # custom NIfTI image
    parcellation_labels=aal_labels  # parcel names (optional)
)
nsp_custom.fit()
print(f"Parcellated with Destrieux: {nsp_custom.get_y().shape[1]} parcels")

[fetch_atlas_aal] Dataset found in /Users/llotter/nilearn_data/aal_SPM12
AAL atlas: 117 labels
INFO | 01/06/26 15:03:18 | nispace.api: *** NiSpace.fit() - Data extraction and preparation. ***


INFO | 01/06/26 15:03:18 | nispace.core.parcellation: Building Parcellation from path / image.


INFO | 01/06/26 15:03:18 | nispace.core.parcellation: Parcellation space: 'MNI152NLin6Asym'.


/var/folders/6n/h4150p8d5gz5kbnqv5_406940000gp/T/ipykernel_28706/3862712985.py:6: DeprecationWarning: Starting in version 0.13, the default fetched mask will beAAL 3v2 instead.
  aal = nilearn_datasets.fetch_atlas_aal()


INFO | 01/06/26 15:03:18 | nispace.core.parcellation: Parcellation 'None': validation passed.


...

Parcellating (1 proc): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 25.80it/s]

INFO | 01/06/26 15:03:20 | nispace.api: Got 'y' data for 1 x 116 parcels.


INFO | 01/06/26 15:03:20 | nispace.api: Z-standardizing 'X' data.


INFO | 01/06/26 15:03:20 | nispace.api: Returning Y dataframe: 
| Y_TRANSFORM | 
| False       | 


Parcellated with Destrieux: 116 parcels


When using a custom parcellation, precomputed spin/distance matrices are not available. NiSpace computes a distance matrix from parcel centroids on the fly when you run permutation tests — this takes a bit longer the first time.

## Storage and reproducibility

By default, all data lands in `~/nispace-data/`. Change this with:

```python
import os
os.environ["NISPACE_DATA_DIR"] = "/your/path"
```

Every downloaded file is verified against a SHA-256 hash manifest — corrupted or incomplete downloads are detected and re-fetched automatically. See the [Data Management](../data_management.rst) page for details on versioning and reproducibility.

## Summary

| Function | Purpose |
|----------|---------|
| `fetch_reference(dataset, parcellation, collection)` | Load curated reference maps (PET, mRNA, ENIGMA, …) |
| `fetch_example("anorexianervosa", parcellation)` | Load the group-comparison example dataset |
| `fetch_parcellation(name)` | Fetch a built-in parcellation as a multi-space object |
| `NiSpace(parcellation=nifti_img)` | Use any NIfTI as a custom parcellation |
| `get_distance_matrix(parcellation)` | Load/compute geodesic distance matrix |

Next: [Notebook 4](intro04_imaging_phenotypes.ipynb) shows how to compute group-level effect sizes from individual subject data.